# Tabular Q-Learning - Single Agent

## Transport Task in Grid World

Author: Mostafiz Rahman

Assignment: Part 1 - Tabular Q-Learning (Single Agent)

Objective: 
 Build a Q-Learning agent from scratch that learns to pick up an item from location A and deliver it to Location B in an nxn grid world.

## Problem Statement

The goal of this projects is to train a single agent using Tabular Q-Learning.

The agent must:

1. Start from a random position.
2. Find the pickup location (A).
3. Automatically pick up the item.
4. Deliver the item to the delivery location (B).
5. Learn the shortest path through training.

In [10]:
import random
import numpy as np

## Environment Setup

In this section, we create the grid world environment that will be used for training the Q-Learning agent.

In [3]:
GRID_SIZE = 5
has_package = False
# empty grid
grid = []

# create grid automatically
for i in range(GRID_SIZE):
    row = ["🟧"] * GRID_SIZE
    grid.append(row)
    
# print Grid
for row in grid:
    for value in row:
        print(value, end=" ")
    print()
    
# Robot Position added (Random)
robot_row = random.randint(0, GRID_SIZE -1)
robot_col = random.randint(0, GRID_SIZE -1)
    
# Pickup Position added (Random)
pickup_row = random.randint(0, GRID_SIZE -1)
pickup_col = random.randint(0, GRID_SIZE -1)
    
# Delivery Position added (Fixed)
delivery_row = GRID_SIZE -1
delivery_col = GRID_SIZE -1

# ==========>>>>>>>>> collision handling <<<<<<<<<<<<<================

# if Robot position and pickup position are same
while robot_row == pickup_row and robot_col == pickup_col:
    pickup_row = random.randint(0, GRID_SIZE - 1)
    pickup_col = random.randint(0, GRID_SIZE - 1)

# if Robot position and delivery position are same
while robot_row == delivery_row and robot_col == delivery_col:
    robot_row = random.randint(0, GRID_SIZE - 1)
    robot_col = random.randint(0, GRID_SIZE - 1)

# if pickup position and delivery position are same
while pickup_row == delivery_row and pickup_col == delivery_col:
    pickup_row = random.randint(0, GRID_SIZE - 1)
    pickup_col = random.randint(0, GRID_SIZE - 1)

# set the robot imoje
grid[robot_row][robot_col] = "🤖"
# set the pickup imoje
grid[pickup_row][pickup_col] = "📦"
# set the robot imoje
grid[delivery_row][delivery_col] = "🏠"
    
# grid print
for row in grid:
    for value in row:
        print(value, end=" ")
    print()

🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🤖 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 📦 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🏠 


## Agent Movement

In [4]:


while True:
    # Grid print
    for row in grid:
        for value in row:
           print(value, end=" ")
        print()
    
    # save previous position
    previous_row = robot_row
    previous_col = robot_col
    
    # User input for direction
    move = input("Enter direction: ").lower()
    print("Move: ", move)
    
    if move == "exit":
        print("Program End")
        break
    
    # remove robot for prev
    grid[robot_row][robot_col] = "🟧"
    
    # move robot
    if move == "left":
        robot_col -= 1
        
    elif move =="right":
        robot_col += 1
        
    elif move == "up":
        robot_row -= 1
        
    elif move == "down":
        robot_row += 1

    else:
        print("❌ Invalid Direction")
        
    # Boundary Check
    robot_row = max(0, min(robot_row, GRID_SIZE -1))
    robot_col = max(0, min(robot_col, GRID_SIZE -1))
    
    # Automatic Pickup
    if robot_row == pickup_row and robot_col == pickup_col:
       print("📦 Package Picked Up!")
       has_package = True
       
    # Automatic Delivery
    if has_package and robot_row == delivery_row and robot_col == delivery_col:
       print("🎉 Delivery Complete!")
       grid[robot_row][robot_col] = "🤖"
       for row in grid:
           for value in row:
               print(value, end=" ")
           print()
       break
    
    grid[robot_row][robot_col] = "🤖"
    if not has_package:
        grid[pickup_row][pickup_col] = "📦"
    grid[delivery_row][delivery_col] = "🏠"

🟧 🟧 🟧 🟧 🟧 
🤖 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 📦 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🏠 
Move:  exit
Program End


## Reward Structure

### Reward Design

The agent receives rewards and penalties based on its actions.

| Action | Reward |
|--------|--------|
| Pick up package | +10 |
| Deliver package | +100 |
| Normal movement | -1 |
| Invalid move / Hit boundary | -5 |

In [71]:
PICKUP_REWARD = 10
DELIVERY_REWARD = 100
STEP_PENALTY = -1
BOUNDARY_PENALTY = -5

## State Representation

In [5]:
# Current state

state = (
    (robot_row, robot_col),
    (pickup_row, pickup_col),
    has_package
)

print(state)

((1, 0), (2, 3), False)


## Q-Table
### What is a Q-Table?

A Q-Table stores the expected reward for taking each action from every possible state.

Rows = States

Columns = Actions

Values = Q-Values

In [7]:
# Action 
# Q-learing use this list

ACTIONS = [
    "up",
    "down",
    "left",
    "right",
    "up-left",
    "up-right",
    "down-left",
    "down-right",
]


## Initialize Q-Table

In [19]:
# Total Possible States
TOTAL_STATES = GRID_SIZE * GRID_SIZE * GRID_SIZE * GRID_SIZE * 2

# Create Q-Table
q_table = np.zeros((TOTAL_STATES, len(ACTIONS)))

print("Q-table shape: ", q_table.shape)

Q-table shape:  (1250, 8)


## Q-Learning Formula
Q(s,a) = Q(s,a) + α × [R + γ × max(Q(s',a')) − Q(s,a)]

### Q-Learning Formula Components

- **s** = Current State
- **a** = Current Action
- **R** = Reward
- **α (Alpha)** = Learning Rate
- **γ (Gamma)** = Discount Factor
- **s'** = Next State
- **max(Q(s',a'))** = Best Future Q-Value



### Learning Rate (α)
How quickly will robots learn new things?